[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S06/S06_04_metricas_validacion.ipynb)

# S06 · 04 — scikit-learn básico: métricas y validación

**Tiempo estimado:** 65 min · **Sprint 6** · M3 (UD 3.1 y 3.2)

**Contexto TurisData.** Un modelo que "acierta el 90 %" puede ser inútil o, peor, engañoso.
Antes de comparar modelos hay que saber **cómo medirlos honestamente**: con qué métrica,
con qué partición y con qué *baseline* de referencia. Trabajamos con `reservas_turisdata.csv`
(6000 reservas; objetivo `cancelada`).

**Al terminar sabrás:**
1. Usar la API de scikit-learn (`fit`, `predict`, `score`) y un modelo *baseline*.
2. Calcular métricas de regresión y clasificación.
3. Detectar sobreajuste con curvas de train/test (sesgo-varianza).
4. Aplicar validación cruzada, detectar **fuga de datos** y usar una **partición temporal**.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold,
                                     TimeSeriesSplit, GridSearchCV)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)

SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "mes_llegada"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]


def hacer_preprocesado(num=columnas_num, cat=columnas_cat):
    """Receta estándar: imputar + escalar (num) e imputar + one-hot (cat)."""
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ])


X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
print("Tasa de cancelación:", y.mean().round(3))

## 1. La API de scikit-learn y el *baseline*

**Idea.** Todos los modelos siguen el mismo patrón: `modelo.fit(X_train, y_train)` aprende,
`modelo.predict(X_nuevo)` predice y `modelo.score(X, y)` mide con la métrica por defecto.
Antes de nada, un **baseline**: un modelo tonto (p. ej., "predecir siempre la clase más
frecuente"). Si tu modelo no lo supera claramente, no aporta nada.

In [ ]:
tonto = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Exactitud del baseline:", round(tonto.score(X_test, y_test), 3))

**Ejercicio 1.** Crea `logistica`, un `Pipeline` con `hacer_preprocesado()` y
`LogisticRegression(max_iter=1000)`. Ajústalo con train y guarda en `acc_logistica`
la exactitud en test y en `mejora` la diferencia con `tonto.score(...)`.

In [ ]:
# TODO: Pipeline([("prep", ...), ("clf", ...)]).fit(...); logistica.score(X_test, y_test)
...
assert acc_logistica > 0.65, "La regresión logística debería superar 0,65 de exactitud"
assert mejora > 0.05, "Debería mejorar al baseline en más de 5 puntos"

## 2. Métricas: ¿cómo de bueno es?

**Clasificación** (predecir una clase). Con `positivo = cancelada`:
- **Exactitud (accuracy):** % de aciertos. Engaña si las clases están desbalanceadas.
- **Precisión:** de las reservas que decimos "se cancelará", ¿cuántas se cancelan de verdad?
- **Recall (sensibilidad):** de las que se cancelan, ¿cuántas detectamos?
- **F1:** media armónica de precisión y recall.

**Regresión** (predecir un número, p. ej. `precio_noche`):
- **MAE:** error absoluto medio (en euros, fácil de explicar).
- **RMSE:** raíz del error cuadrático medio (castiga más los errores grandes).
- **R²:** proporción de la variación explicada (1 = perfecto; 0 = como predecir la media).

In [ ]:
pred = logistica.predict(X_test)
print(f"Exactitud {accuracy_score(y_test, pred):.3f} · Precisión {precision_score(y_test, pred):.3f} "
      f"· Recall {recall_score(y_test, pred):.3f} · F1 {f1_score(y_test, pred):.3f}")

**Ejercicio 2.** Entrena una regresión lineal para predecir `precio_noche` usando solo
`tipo_habitacion` y `mes_llegada` (usa `hacer_preprocesado(num=["mes_llegada"], cat=["tipo_habitacion"])`).
Con los datos de test, guarda `mae`, `rmse` y `r2`.

In [ ]:
# TODO: el objetivo ahora es df["precio_noche"]; mean_squared_error(...) ** 0.5 para el RMSE
...
assert 5 < mae < 30, "El MAE debería estar entre 5 y 30 euros. Revisa el objetivo"
assert rmse >= mae, "El RMSE siempre es mayor o igual que el MAE"
assert 0.5 < r2 < 1, "R² debería ser alto: el tipo de habitación explica gran parte del precio"

## 3. Sobreajuste y sesgo-varianza

**Idea.** Un modelo muy flexible puede **memorizar** el entrenamiento (*sobreajuste*: alta
varianza) y fallar con datos nuevos. Uno demasiado simple no captura el patrón (*infraajuste*:
alto sesgo). La señal clásica: la exactitud en **train sigue subiendo** mientras la de **test
se estanca o baja**. Lo vemos con un árbol de profundidad creciente.

In [ ]:
prep_arbol = hacer_preprocesado()
Xtr_p = prep_arbol.fit_transform(X_train)
Xte_p = prep_arbol.transform(X_test)
profundidades = range(1, 21)
acc_tr, acc_te = [], []
for p in profundidades:
    arbol = DecisionTreeClassifier(max_depth=p, random_state=SEMILLA).fit(Xtr_p, y_train)
    acc_tr.append(arbol.score(Xtr_p, y_train))
    acc_te.append(arbol.score(Xte_p, y_test))
plt.figure(figsize=(8, 4))
plt.plot(profundidades, acc_tr, marker="o", label="Entrenamiento")
plt.plot(profundidades, acc_te, marker="s", label="Prueba")
plt.title("Sobreajuste: profundidad del árbol vs. exactitud")
plt.xlabel("Profundidad máxima del árbol")
plt.ylabel("Exactitud")
plt.legend()
plt.show()

**Ejercicio 3.** A partir de las listas `acc_te` y `profundidades`, guarda en `mejor_profundidad`
la profundidad con mejor exactitud en test, y en `brecha_maxima` la diferencia
(train − test) con profundidad 20 (`acc_tr[-1] - acc_te[-1]`).

In [ ]:
# TODO: np.argmax(acc_te) te da la posición; las profundidades empiezan en 1
...
assert 1 <= mejor_profundidad <= 20
assert brecha_maxima > 0.2, "Con profundidad 20 la brecha train-test debería ser grande (sobreajuste)"
print(f"Mejor profundidad en test: {mejor_profundidad} · brecha con profundidad 20: {brecha_maxima:.2f}")

> **Cuidado:** elegir la profundidad mirando el conjunto de **test** ya es una pequeña
> fuga (el test deja de ser un examen sorpresa). Lo correcto es elegir con **validación
> cruzada sobre train** y usar test una sola vez al final.

## 4. Validación cruzada

**Idea.** En lugar de una única partición (que depende de la suerte), se divide train en
*k* trozos (*folds*): se entrena con *k−1* y se valida con el restante, rotando. Obtenemos
*k* puntuaciones: su **media** estima el rendimiento y su **desviación** la estabilidad.
`StratifiedKFold` mantiene la proporción de clases en cada trozo.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
puntuaciones = cross_val_score(logistica, X_train, y_train, cv=cv, scoring="accuracy")
print("Exactitud por fold:", puntuaciones.round(3))
print(f"Media {puntuaciones.mean():.3f} ± {puntuaciones.std():.3f}")

**Ejercicio 4.** Calcula `auc_cv`: el array de puntuaciones de validación cruzada de
`logistica` con `scoring="roc_auc"` y el mismo `cv` (5 folds). Guarda su media en `auc_medio`.

In [ ]:
# TODO: cross_val_score(logistica, X_train, y_train, cv=cv, scoring="roc_auc")
...
assert len(auc_cv) == 5, "Deben ser 5 puntuaciones (5 folds)"
assert 0.65 < auc_medio < 0.95, "El AUC medio debería estar entre 0,65 y 0,95"

## 5. Fuga de datos (*data leakage*)

**Idea.** Hay fuga cuando el modelo usa información que **no tendría en el momento de
predecir** en la vida real. Resultado: métricas fantásticas en el laboratorio y fracaso
en producción. Ejemplos: columnas creadas *después* del desenlace (p. ej. "importe reembolsado"),
estadísticos calculados con todo el dataset o duplicados entre train y test.

Simulamos una columna con fuga: `motivo_baja_sistema`, que el sistema rellena **solo cuando
la reserva se cancela**. Ninguna persona real la conocería al reservar.

In [ ]:
rng = np.random.default_rng(SEMILLA)
X_fuga = X.copy()
X_fuga["reembolso_emitido"] = np.where(y == 1, rng.choice([1, 1, 1, 0], len(y)), 0)  # casi = cancelada
prep_fuga = hacer_preprocesado(num=columnas_num + ["reembolso_emitido"])
modelo_fuga = Pipeline([("prep", prep_fuga), ("clf", LogisticRegression(max_iter=1000))])
auc_fuga = cross_val_score(modelo_fuga, X_fuga, y, cv=cv, scoring="roc_auc").mean()
auc_real = cross_val_score(logistica, X, y, cv=cv, scoring="roc_auc").mean()
print(f"AUC sin la columna: {auc_real:.3f} · AUC con la columna que 'delata' el resultado: {auc_fuga:.3f}")

**Ejercicio 5.** De la lista `candidatas`, indica en `columnas_con_fuga` (lista) cuáles **no
se conocerían en el momento de hacer la reserva** y por tanto provocarían fuga.

In [ ]:
candidatas = {
    "antelacion_dias": "días entre reservar y llegar (se conoce al reservar)",
    "importe_devuelto": "euros reembolsados al cliente tras cancelar",
    "tarifa": "flexible o no reembolsable (se elige al reservar)",
    "fecha_cancelacion": "fecha en que el cliente canceló (vacía si no canceló)",
    "cancelaciones_previas": "cancelaciones de reservas anteriores del cliente",
}
# TODO: pregúntate: ¿existe esta información antes de saber si cancela?
...
assert set(columnas_con_fuga) == {"importe_devuelto", "fecha_cancelacion"}, "Piensa en qué datos solo existen DESPUÉS de cancelar"

## 6. Partición temporal

**Idea.** Cuando los datos tienen fecha, el modelo se usará para **predecir el futuro**.
Mezclar al azar deja que el modelo "vea" reservas posteriores a las que evalúa: demasiado
optimista. La partición temporal entrena con el pasado y evalúa con lo más reciente.
(Con `TimeSeriesSplit`, varios cortes sucesivos.)

In [ ]:
df_ord = df.sort_values("fecha_reserva").reset_index(drop=True)
corte = int(len(df_ord) * 0.8)
Xo, yo = df_ord[columnas_num + columnas_cat], df_ord["cancelada"]
mod_t = Pipeline([("prep", hacer_preprocesado()), ("clf", LogisticRegression(max_iter=1000))])
mod_t.fit(Xo.iloc[:corte], yo.iloc[:corte])
print("Fecha de corte:", df_ord.loc[corte, "fecha_reserva"].date())
print("Exactitud en el futuro (partición temporal):", round(mod_t.score(Xo.iloc[corte:], yo.iloc[corte:]), 3))

**Ejercicio 6.** Usa `TimeSeriesSplit(n_splits=4)` sobre `Xo`, `yo` (ya ordenados por fecha)
para calcular `auc_temporal`: array con la AUC ROC de cada corte (`scoring="roc_auc"`).

In [ ]:
# TODO: cross_val_score(mod_t, Xo, yo, cv=TimeSeriesSplit(n_splits=4), scoring="roc_auc")
...
assert len(auc_temporal) == 4, "Debe haber 4 puntuaciones"
assert auc_temporal.min() > 0.6, "Cada corte debería superar 0,6 de AUC"

## 7. Buscar la mejor configuración: `GridSearchCV`

Un **hiperparámetro** es un ajuste que decidimos nosotros (no lo aprende el modelo), como
la `C` de la regresión logística. `GridSearchCV` prueba todas las combinaciones con
validación cruzada y se queda con la mejor. Se aplica **solo sobre train**.

In [ ]:
rejilla = {"clf__C": [0.01, 0.1, 1, 10]}
busqueda = GridSearchCV(logistica, rejilla, cv=cv, scoring="roc_auc").fit(X_train, y_train)
print("Mejor C:", busqueda.best_params_, "· AUC en validación:", round(busqueda.best_score_, 3))

## 8. Mini-reto: tabla de resultados honesta

Crea `resultados`, un DataFrame con una fila por modelo (`"baseline"`, `"logistica"`) y las
columnas `auc_cv_media` y `auc_cv_desv` (validación cruzada de 5 folds sobre train, `roc_auc`).
Para el baseline usa `DummyClassifier(strategy="prior")`. Añade una frase en `conclusion`
que diga si compensa el modelo frente al baseline.

In [ ]:
# TODO: construye un dict {modelo: puntuaciones} y luego pd.DataFrame
...
assert list(resultados.index) == ["baseline", "logistica"]
assert abs(resultados.loc["baseline", "auc_cv_media"] - 0.5) < 0.01, "El baseline debería dar AUC ≈ 0,5"
assert resultados.loc["logistica", "auc_cv_media"] > 0.65
assert isinstance(conclusion, str) and len(conclusion) > 20
resultados

## 9. Para reflexionar

1. Un modelo acierta el 54 % de las cancelaciones y el baseline acierta el 54 %. ¿Qué concluyes?
2. ¿Por qué una buena puntuación en el propio conjunto de entrenamiento no demuestra nada?
3. Si TurisData quiere usar el modelo para reservas **futuras**, ¿qué partición es más honesta y por qué?
4. ¿Qué fugas podría tener el dataset real del cliente que no se ven a primera vista?

## 10. Resumen: qué has aprendido
- Siempre un **baseline** antes de un modelo "de verdad".
- Elige la métrica según el problema (MAE/RMSE/R² en regresión; precisión, recall, F1, AUC en clasificación).
- Sobreajuste = train brillante, test pobre. Se detecta con curvas y validación cruzada.
- Fuga de datos: la información del futuro (o del test) nunca debe colarse en el ajuste.
- Con fechas, valida en el tiempo.